# Metadata

BDF writes rich metadata alongside each dataset (as JSON-LD embedded in an HTML landing page and/or sidecar files), so data are self-describing and web-discoverable. 

## Design Principles
BDF favors schema.org for maximum interoperability on the semantic web (and rich presentation in Dataset Search), while also supporting CSVW for precise tabular schemas. Quantity meanings are grounded in the BDF application ontology to keep terminology consistent across tools and datasets.

## Content

BDF metadata covers three main categories:

1. **Bibliographic (who/what/when).** This includes information about what the dataset is and who made it. The purpose of bibliographic metadata is to enable proper citations, credit, reproducibility, and searching. It features fields like:  

    - title, description, keywords  
    - creators and contributors  
    - date, version, and license  
    - provenance  

2. **Content (what's in the table).** This describes the quantities in your dataset so tools and search engines can understand them. BDF supports descriptions of table quantities using both schema.org and csvw markup. This mapping is handled automatically by BDF. 

3. **Distribution (where and how to get the file).** This describes the actual downloadable artifact(s). It includes files like:

    - file URL(s), media type (e.g., text/csv, application/parquet)  
    - size/checksums (optional), content variants (raw/processed)  
    - landing page vs. direct download (schema.org/DataDownload)  

In [ ]:
import bdf
from bdf.metadata import Dataset, Creator, DataDownload, Battery

In [ ]:
# Read the raw source data and display the header
# This export stores its time columns in milliseconds under seconds headers;
# bdf.read detects the mismatch against the recorded timestamps and raises
# unless we explicitly opt into the repair with reconcile_time=True.
df, source_meta = bdf.read(
    "https://zenodo.org/records/17295469/files/FZJ__INR21700__20250606__HPPC__25degC__Digatron.csv",
    reconcile_time=True,
)
df = df.to_pandas()
df.head()

In [ ]:
meta = Dataset(
    title="Digatron INR21700 HPPC",
    creators=[Creator(name="Example Creator", orcid="0000-0002-0000-0010", given_name="Example", family_name="Creator", affiliation="Your Lab")],
    description="HPPC characterization of an INR21700 cell at 25°C on a Digatron cycler.",
    keywords=["li-ion", "inr21700", "hppc"],
    license="CC-BY-4.0",
    url="https://zenodo.org/records/17295469",
    version="1.0.0",
    publication_date="2025-06-06",
)

dist = DataDownload(
    url="https://zenodo.org/records/17295469/files/FZJ__INR21700__20250606__HPPC__25degC__Digatron.csv",
    name="Digatron CSV file",
    encoding_format="text/csv",
    description="Primary CSV export from Digatron cycler."
)

battery = Battery(
    id="g20m7",
    model = "G20M7",
    manufacturer = "Google", 
    iec_code = "ICP6/65/75",
    form_factor = "prismatic",
    nominal_voltage_v = 3.90,
    rated_capacity_ah = 4.835,
    mass_g = 63,
    volume_l = 0.02925,
    pe_materials = ["cobalt"],
    ne_materials = ["graphite"]
)

In [ ]:
# Save the metadata to JSON-LD

meta.save_jsonld(
    "./out/metadata/dataset.schemaorg.jsonld",
    dataset_uri="https://doi.org/10.5281/zenodo.16994937#digatron-csv-li-ion-hppc",
    identifier="digatron-csv-li-ion-hppc",
    distributions=[dist],
    df=df
)

In [ ]:
# Save the metadata as rich results html for Google / Semantic Web integration

meta.save_rich_results_html(
    "./out/metadata/dataset.schemaorg.html",
    title="Digatron INR21700 HPPC",
    graphify=True,
    dataset_uri="https://doi.org/10.5281/zenodo.16994937#digatron-csv-li-ion-hppc",
    identifier="digatron-csv-li-ion-hppc",
    distributions=[dist],
    df=df
)

## Field-declared JSON-LD projection

The metadata above is written by hand. The projection layer in `bdf.metadata_projection` takes the
opposite approach: each field declares its own RDF mapping inline as `Annotated[type, Marker(...)]`,
and a single generic walker turns any model into a graph and back. There is no per-model
serialiser to keep in step, and a field carrying no marker is simply not projected.

In [ ]:
from datetime import date
from typing import Annotated, ClassVar

from pydantic import Field

from bdf.metadata_projection import (
    BdfModel,
    NodeRef,
    SameAs,
    Scalar,
    StrList,
    Typed,
    from_compact_jsonld,
    to_compact_jsonld,
)


class Lab(BdfModel):
    """An organisation, referenced from the note below."""

    _rdf_type: ClassVar[str | None] = "schema:Organization"

    name: Annotated[str, Scalar("schema:name")]
    ror: Annotated[str | None, SameAs(prefix="https://ror.org/")] = None


class Note(BdfModel):
    """A minimal dataset note. Every projected fact is declared on its field."""

    _rdf_type: ClassVar[str | None] = "schema:Dataset"

    name: Annotated[str, Scalar("schema:name")]
    published: Annotated[date | None, Typed("schema:datePublished", datatype="xsd:date")] = None
    keywords: Annotated[list[str], StrList("schema:keywords")] = Field(default_factory=list)
    publisher: Annotated[str | Lab | None, NodeRef(role="publisher", term="schema:publisher")] = None
    internal_note: str | None = None  # no marker, so never projected


In [ ]:
note = Note(
    name="HPPC at 25 degC",
    published=date(2026, 7, 27),
    keywords=["hppc", "inr21700"],
    publisher=Lab(name="Battery Data Alliance", ror="052gg0110"),
    internal_note="never leaves Python",
)

# The document base comes from the document's own name, so every identifier is
# relative to the file that carries it and no subject is ever a blank node.
text = to_compact_jsonld(note, base="notes.jsonld")
print(text)

Three things to notice in that output. The publisher is its own node at
`notes.jsonld#record/publisher` rather than an anonymous blob, so another document can point at
it. The two keywords are two direct `schema:keywords` edges, not an ordered RDF collection —
RDF asserts no order over repeated properties, so neither does BDF. And `internal_note` is
absent: it carries no marker.

In [ ]:
# Reading back is the same walker in reverse.
restored = from_compact_jsonld(Note, text)

print(restored.name)
print(restored.publisher)
print(restored.internal_note)  # None: it was never in the document

### Graphs are compared as graphs

Two serialisations of the same metadata differ in triple order and formatting, so comparing
their text answers the wrong question. Compare the graphs instead — `rdflib.compare.isomorphic`
does it, and BDF's round-trip contract is stated in exactly those terms: RDF equivalence, never
byte identity.


In [ ]:
from rdflib import Graph, Literal, URIRef
from rdflib.compare import isomorphic

# N-Triples requires absolute IRIs, so this comparison uses an absolute document
# IRI rather than the relative one a sidecar carries.
subject = URIRef("https://example.org/notes.jsonld#record")
graph = note.to_graph(subject=subject)

# The same graph in two syntaxes: different bytes, same facts.
turtle = graph.serialize(format="turtle")
ntriples = graph.serialize(format="nt")

reparsed_turtle = Graph().parse(data=turtle, format="turtle")
reparsed_ntriples = Graph().parse(data=ntriples, format="nt")

print("same bytes: ", turtle == ntriples)
print("same graph: ", isomorphic(reparsed_turtle, reparsed_ntriples))


In [ ]:
# One changed literal is still caught, so the comparison is not vacuous.
edited = Graph()
edited += graph
edited.remove((subject, URIRef("https://schema.org/name"), Literal("HPPC at 25 degC")))
edited.add((subject, URIRef("https://schema.org/name"), Literal("HPPC at 45 degC")))

print("same graph: ", isomorphic(graph, edited))
